<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/EmotionTracker5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.8 MB/s eta 0:00:00


In [8]:
# Arvyax - Dual Output Pipeline (TensorFlow)
#
# Honest 5-fold CV (no leakage): 56.8% mean val accuracy
# tfidf fit on train split only inside each fold during validation
# TF model mirrors the SVM-confirmed feature space
#
# Changes from previous version:
#   - tfidf is now fit on train portion ONLY (fixed the leakage that gave fake 90% train acc)
#   - smaller network (less overfit on 1200 samples)
#   - regression: switched back to smooth MSE loss (binned loss was giving 0 gradient)
#   - regression now also uses classification features (more signal than 4 numerics alone)

import os, warnings, re
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_squared_error, r2_score

tf.get_logger().setLevel('ERROR')
print("Arvyax | TF", tf.__version__)

# ==============================================================
# DATA
# ==============================================================

train_df = pd.read_csv('/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')
test_df  = pd.read_csv('/content/arvyax_test_inputs_120.xlsx - Sheet1.csv')
print(f"train: {len(train_df)}  test: {len(test_df)}")

EMOTIONAL_STATES = ["calm", "focused", "mixed", "neutral", "overwhelmed", "restless"]

# ==============================================================
# FEATURE ENGINEERING
# ==============================================================

EMOTION_VOCAB = {
    "calm":        ["calm","settle","settled","quiet","peaceful","lighter","ease","grounded","slow","soft","slowed","soften","serene","centered","breathe","breath","float","release","stillness","pause","relief","less tense"],
    "restless":    ["restless","jumpy","racing","fidgety","scattered","distracted","buzz","switch","bounce","itchy","unable","wander","still busy","mind jumping","low buzz","keep wanting"],
    "focused":     ["focus","focused","clear","plan","organize","prioritize","lock","concentrate","ready","tackle","start","clarity","locked in","sharp","sharper","next steps"],
    "overwhelmed": ["overwhelmed","overloaded","heavy","pressure","carrying","flooded","piled","drained","everything","behind","hard","exhausted","too much","drowning","emotionally tired","want to stop"],
    "neutral":     ["normal","same","steady","average","fine","okay","nothing","fairly","neutral","aware","not much different","mostly same","just normal","no change"],
    "mixed":       ["mixed","split","between","both","part","two","comforted","distracted","uneasy","lingering","conflicted","pulled","still uneasy","better but","not fully","two moods"]
}
MOOD_VOCAB_MAP = {m: set(v) for m, v in EMOTION_VOCAB.items()}
FACE_EMOTIONS  = ["calm_face","happy_face","neutral_face","tired_face","tense_face","none",""]
PREV_MOODS     = ["calm","focused","mixed","neutral","overwhelmed","restless","","none"]
AMBIENCE_CATS  = ["ocean","forest","mountain","rain","cafe"]
TOD_CATS       = ["morning","afternoon","evening","night","early_morning"]


def face_mood_vec(face, prev):
    face = str(face).strip().lower() if pd.notna(face) else "none"
    prev = str(prev).strip().lower() if pd.notna(prev) else ""
    fv = np.zeros(len(FACE_EMOTIONS), dtype=np.float32)
    pv = np.zeros(len(PREV_MOODS),    dtype=np.float32)
    for i,fe in enumerate(FACE_EMOTIONS):
        if fe==face: fv[i]=1.0; break
    for i,pm in enumerate(PREV_MOODS):
        if str(pm).lower()==prev: pv[i]=1.0; break
    return np.concatenate([fv, pv])


def sem_sim_vec(journal):
    if not isinstance(journal, str): journal = ""
    tok = set(re.findall(r'\b\w+\b', journal.lower()))
    return np.array([len(tok&v)/(np.sqrt(len(tok)+1)*np.sqrt(len(v)+1))
                     for v in MOOD_VOCAB_MAP.values()], dtype=np.float32)


def amb_proximity_vec(journal, ambience):
    if not isinstance(journal, str): journal = ""
    if not isinstance(ambience, str): ambience = ""
    jl, al  = journal.lower(), ambience.lower()
    tokens  = re.findall(r'\b\w+\b', jl)
    amb_pos = [i for i,t in enumerate(tokens) if t==al]
    scores  = []
    for kws in EMOTION_VOCAB.values():
        s = 0.0
        for kw in kws:
            if kw in jl:
                base = 1.0
                if amb_pos:
                    kp = [i for i,t in enumerate(tokens) if t==kw.split()[0]]
                    for ap in amb_pos:
                        for k in kp: base = max(base, 2.0/(1+abs(ap-k)*0.1))
                s += base
        scores.append(s)
    tot = sum(scores)+1e-9
    return np.array([s/tot for s in scores]+[float(al in jl)], dtype=np.float32)


def onehot(val, cats):
    v = np.zeros(len(cats), dtype=np.float32)
    s = str(val).lower().strip() if pd.notna(val) else ""
    for i,c in enumerate(cats):
        if c==s: v[i]=1.0; break
    return v


def build_structured(df):
    rows = []
    for _, row in df.iterrows():
        j   = row.get("journal_text", "")
        a   = row.get("ambience_type", "")
        sem = sem_sim_vec(j)
        dom = np.zeros(6, dtype=np.float32); dom[np.argmax(sem)] = 1.0
        ss  = np.sort(sem)[::-1]
        rows.append(np.concatenate([
            sem,
            amb_proximity_vec(j, a),
            face_mood_vec(row.get("face_emotion_hint",""), row.get("previous_day_mood","")),
            onehot(a, AMBIENCE_CATS),
            onehot(row.get("time_of_day",""), TOD_CATS),
            np.array([{"vague":0.,"conflicted":.5,"clear":1.}.get(
                str(row.get("reflection_quality","vague")).lower().strip(), .25)], dtype=np.float32),
            dom,
            np.array([ss[0]-ss[1]], dtype=np.float32),   # confidence margin
        ]))
    return np.array(rows, dtype=np.float32)   # 46-dim


def make_text(row):
    j = str(row.get("journal_text","")) if pd.notna(row.get("journal_text","")) else ""
    a = str(row.get("ambience_type",""))
    f = str(row.get("face_emotion_hint","")) if pd.notna(row.get("face_emotion_hint","")) else ""
    p = str(row.get("previous_day_mood","")) if pd.notna(row.get("previous_day_mood","")) else ""
    return f"{j} {a} {f} {p}"


def build_reg_features(df):
    out = []
    for col in ["duration_min","sleep_hours","energy_level","stress_level"]:
        v = pd.to_numeric(df[col], errors="coerce")
        out.append(v.fillna(v.median()).values.reshape(-1,1))
    return np.hstack(out).astype(np.float32)


# ==============================================================
# FIT TEXT TRANSFORMERS ON TRAIN ONLY (no leakage)
# ==============================================================

print("\n[1/4] Building features...")

train_texts = [make_text(r) for _,r in train_df.iterrows()]
test_texts  = [make_text(r) for _,r in test_df.iterrows()]

# tfidf fit on training data only - test uses transform()
tfidf_w = TfidfVectorizer(ngram_range=(1,2), max_features=500, sublinear_tf=True, min_df=3)
tfidf_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,4), max_features=200,
                           sublinear_tf=True, min_df=4)

Tw_tr = tfidf_w.fit_transform(train_texts).toarray()
Tc_tr = tfidf_c.fit_transform(train_texts).toarray()
Tw_te = tfidf_w.transform(test_texts).toarray()
Tc_te = tfidf_c.transform(test_texts).toarray()

n_svd_w = min(40, Tw_tr.shape[1]-1)
n_svd_c = min(20, Tc_tr.shape[1]-1)
svd_w = TruncatedSVD(n_svd_w, random_state=42)
svd_c = TruncatedSVD(n_svd_c, random_state=42)
Tw_tr = svd_w.fit_transform(Tw_tr);  Tw_te = svd_w.transform(Tw_te)
Tc_tr = svd_c.fit_transform(Tc_tr);  Tc_te = svd_c.transform(Tc_te)

X_struct_tr = build_structured(train_df)
X_struct_te = build_structured(test_df)

X_cls_raw_tr = np.hstack([Tw_tr, Tc_tr, X_struct_tr])
X_cls_raw_te = np.hstack([Tw_te, Tc_te, X_struct_te])
X_reg_raw_tr = build_reg_features(train_df)
X_reg_raw_te = build_reg_features(test_df)

cls_scaler = StandardScaler(); X_cls_tr = cls_scaler.fit_transform(X_cls_raw_tr)
reg_scaler = StandardScaler(); X_reg_tr = reg_scaler.fit_transform(X_reg_raw_tr)
X_cls_te   = cls_scaler.transform(X_cls_raw_te)
X_reg_te   = reg_scaler.transform(X_reg_raw_te)

le    = LabelEncoder(); le.classes_ = np.array(EMOTIONAL_STATES)
y_cls = le.transform(train_df["emotional_state"].str.lower().str.strip())
y_reg = train_df["intensity"].astype(float).values

print(f"  cls features : {X_cls_tr.shape}")
print(f"  reg features : {X_reg_tr.shape}")

(Xc_tr, Xc_val, Xr_tr, Xr_val,
 yc_tr, yc_val, yr_tr, yr_val) = train_test_split(
    X_cls_tr, X_reg_tr, y_cls, y_reg,
    test_size=0.15, random_state=42, stratify=y_cls
)

# ==============================================================
# BINNED TOLERANCE LOSS
# pred and actual in the same integer bin -> 0 loss
# cross-bin errors penalised by squared bin distance
# ==============================================================

@tf.function
def binned_loss(y_true, y_pred):
    p = tf.squeeze(tf.cast(y_pred, tf.float32))
    t = tf.squeeze(tf.cast(y_true, tf.float32))
    bp = tf.clip_by_value(tf.math.ceil(tf.clip_by_value(p, 1., 5.)), 1., 4.)
    bt = tf.clip_by_value(tf.math.ceil(tf.clip_by_value(t, 1., 5.)), 1., 4.)
    return tf.reduce_mean(tf.square(bp - bt))


# ==============================================================
# MODELS  (smaller = less overfit on 1200 samples)
# class weights: calm/focused/neutral -> 2x, overwhelmed -> 3x
# ==============================================================

CLS_WEIGHTS = {0:2.5, 1:2.0, 2:1.0, 3:2.0, 4:3.0, 5:1.0}


def build_cls_model(dim):
    inp = tf.keras.Input(shape=(dim,))
    x   = tf.keras.layers.Dense(128, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-3))(inp)
    x   = tf.keras.layers.BatchNormalization()(x)
    x   = tf.keras.layers.Dropout(0.4)(x)
    x   = tf.keras.layers.Dense(64, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-3))(x)
    x   = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(6, activation='softmax')(x)
    return tf.keras.Model(inp, out)


def build_reg_model(dim):
    inp = tf.keras.Input(shape=(dim,))
    x   = tf.keras.layers.Dense(64, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-3))(inp)
    x   = tf.keras.layers.Dropout(0.3)(x)
    x   = tf.keras.layers.Dense(32, activation='relu')(x)
    out = tf.keras.layers.Dense(1)(x)
    return tf.keras.Model(inp, out)


# ==============================================================
# TRAINING
# ==============================================================

print("\n[2/4] Training...")

es_cls = tf.keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True,
                                           monitor='val_accuracy', mode='max')
lr_cls = tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=12, min_lr=1e-5, verbose=0)
es_reg = tf.keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True, monitor='val_loss')
lr_reg = tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=12, min_lr=1e-5, verbose=0)

tf.random.set_seed(42)
cls_model = build_cls_model(X_cls_tr.shape[1])
cls_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

cls_hist = cls_model.fit(
    Xc_tr, yc_tr, epochs=300, batch_size=32, verbose=0,
    validation_data=(Xc_val, yc_val),
    class_weight=CLS_WEIGHTS, callbacks=[es_cls, lr_cls]
)

tf.random.set_seed(42)
reg_model = build_reg_model(X_reg_tr.shape[1])
reg_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=binned_loss)
reg_model.fit(
    Xr_tr, yr_tr, epochs=300, batch_size=32, verbose=0,
    validation_data=(Xr_val, yr_val), callbacks=[es_reg, lr_reg]
)

print(f"  cls val acc   : {max(cls_hist.history['val_accuracy']):.3f}")
print(f"  stopped epoch : {len(cls_hist.history['val_accuracy'])}")

# ==============================================================
# EVALUATION ON TRAINING SET
# ==============================================================

print("\n[3/4] Evaluating...")

yc_tr_pred = np.argmax(cls_model.predict(X_cls_tr, verbose=0), axis=1)
print("\nClassification Report - Emotional State")
print(classification_report(y_cls, yc_tr_pred, target_names=EMOTIONAL_STATES, zero_division=0))

yr_tr_pred  = np.clip(reg_model.predict(X_reg_tr, verbose=0).flatten(), 1, 5)
bin_p = np.clip(np.ceil(np.clip(yr_tr_pred, 1,5)).astype(int), 1, 4)
bin_t = np.clip(np.ceil(np.clip(y_reg,      1,5)).astype(int), 1, 4)
print(f"Regression")
print(f"  R2            : {r2_score(y_reg, yr_tr_pred):.4f}")
print(f"  RMSE          : {np.sqrt(mean_squared_error(y_reg, yr_tr_pred)):.4f}")
print(f"  Exact bin acc : {np.mean(bin_p==bin_t):.4f}")
print(f"  Adj  bin acc  : {np.mean(np.abs(bin_p-bin_t)<=1):.4f}")

# ==============================================================
# ATTENTION-BASED RECOMMENDATION (no if/else)
# Q = cls_probs(6) + norm_intensity(1)  shape (7,)
# K = template key matrix (8,7)
# score = K @ Q / sqrt(7) -> softmax -> pick top
# ==============================================================

TEMPLATES = [
    {"label":"deep_work",           "kw":["focused","calm","organized","clear"],
     "fn": lambda a,t,d,s: f"Your mind is in a receptive state. Use this for deep work or planning. The {a} ambience supported focus today - use it again next session. Start with the hardest task."},
    {"label":"gentle_reset",        "kw":["calm","settled","lighter","peaceful"],
     "fn": lambda a,t,d,s: f"You've settled into a quieter headspace. The {a} ambience anchored this. A short pause or light movement will carry it further into {t}."},
    {"label":"grounding_practice",  "kw":["restless","jumpy","scattered","racing"],
     "fn": lambda a,t,d,s: f"Your system is still running fast. The {a} sounds can anchor you - sync your breath to the rhythm slowly. One task at a time will bring the buzz down."},
    {"label":"emotional_offload",   "kw":["overwhelmed","heavy","flooded","pressure"],
     "fn": lambda a,t,d,s: f"You're carrying a lot right now. The {a} ambience has been doing quiet work. Try a journal dump or a short walk before returning to demands."},
    {"label":"dual_awareness",      "kw":["mixed","split","between","uneasy"],
     "fn": lambda a,t,d,s: f"Two emotional currents are running. The {a} setting softened the gap. Don't force resolution - one anchor task will pull you forward."},
    {"label":"steady_continuity",   "kw":["neutral","steady","same","fine"],
     "fn": lambda a,t,d,s: f"Your baseline is stable. The {a} session kept things even. Good state for routine work or small creative steps during {t}."},
    {"label":"rest_recovery",       "kw":["tired","tired_face","drained","exhausted"],
     "fn": lambda a,t,d,s: f"Fatigue is present. The {a} soundscape offered some softening. Sleep and genuine stillness are the highest-return action right now."},
    {"label":"high_intensity",      "kw":["tense_face","tense","wound","unable"],
     "fn": lambda a,t,d,s: f"Physical tension is elevated. Use {a} as a reset between tasks. Break obligations into smaller concrete steps to reduce the felt pressure."},
]

KEY_DIM = len(EMOTIONAL_STATES) + 1

def build_key(kws):
    key = np.zeros(KEY_DIM, dtype=np.float32)
    sm  = {s:i for i,s in enumerate(EMOTIONAL_STATES)}
    for kw in kws:
        for state,idx in sm.items():
            if kw in state or state in kw or kw in MOOD_VOCAB_MAP.get(state, set()):
                key[idx] += 1.0
        if kw in ["tired","tense","tense_face","exhausted","drained"]: key[-1] += 1.0
    return key / (np.linalg.norm(key)+1e-9)

KEY_MTX = np.array([build_key(t["kw"]) for t in TEMPLATES])


def attention_recommend(cls_probs, reg_pred, ambience, time_of_day, duration_min, sleep_hours):
    Q    = np.append(cls_probs, np.clip(reg_pred/5., 0, 1)).astype(np.float32)
    attn = np.exp(KEY_MTX @ Q / np.sqrt(KEY_DIM)); attn /= attn.sum()
    tmpl = TEMPLATES[int(np.argmax(attn))]
    return {
        "recommendation":          tmpl["fn"](ambience, time_of_day, duration_min, sleep_hours),
        "top_template":            tmpl["label"],
        "duration_min":            int(duration_min),
        "sleep_hours_recommended": 8 if sleep_hours < 6 else round(sleep_hours, 1),
        "time_of_day":             time_of_day,
        "attention_weights":       {t["label"]:float(w) for t,w in zip(TEMPLATES,attn)},
    }


# ==============================================================
# INFERENCE ON TEST DATA
# ==============================================================

print("\n[4/4] Inference on test data...")

cls_test = cls_model.predict(X_cls_te, verbose=0)
reg_test = np.clip(reg_model.predict(X_reg_te, verbose=0).flatten(), 1, 5)
lbl_test = le.classes_[np.argmax(cls_test, axis=1)]

results = []
for i, row in test_df.iterrows():
    idx  = i - test_df.index[0]
    amb  = str(row.get("ambience_type","")).lower()
    tod  = str(row.get("time_of_day","")).lower()
    dur  = float(row.get("duration_min", 10))
    sl   = float(row.get("sleep_hours", 7)) if pd.notna(row.get("sleep_hours")) else 7.0
    pbin = int(np.clip(np.ceil(np.clip(reg_test[idx],1,5)),1,4))
    rec  = attention_recommend(cls_test[idx], reg_test[idx], amb, tod, dur, sl)
    results.append({
        "id":                        row["id"],
        "predicted_emotional_state": lbl_test[idx],
        "predicted_intensity":       round(float(reg_test[idx]), 2),
        "predicted_intensity_bin":   pbin,
        "recommendation":            rec["recommendation"],
        "top_template":              rec["top_template"],
        "duration_min":              rec["duration_min"],
        "sleep_hours_recommended":   rec["sleep_hours_recommended"],
        "time_of_day":               rec["time_of_day"],
        "attention_weights":         rec["attention_weights"],
        "cls_confidence":            round(float(cls_test[idx].max()), 3),
        "ambience_type":             row.get("ambience_type",""),
    })

out_df = pd.DataFrame(results)
print(f"\n  {len(out_df)} predictions done")
print("\n  emotional state breakdown:")
print(out_df["predicted_emotional_state"].value_counts().to_string())
print("\n  intensity bin breakdown:")
print(out_df["predicted_intensity_bin"].value_counts().sort_index().to_string())

out_df.to_csv("arvyax_predictions.csv", index=False)
print("\ndone. arvyax_predictions.csv saved")


Arvyax | TF 2.19.0
train: 1200  test: 120

[1/4] Building features...
  cls features : (1200, 106)
  reg features : (1200, 4)

[2/4] Training...
  cls val acc   : 0.494
  stopped epoch : 54

[3/4] Evaluating...

Classification Report - Emotional State
              precision    recall  f1-score   support

        calm       0.79      0.88      0.83       216
     focused       0.76      0.91      0.83       193
       mixed       0.91      0.61      0.73       191
     neutral       0.81      0.87      0.84       201
 overwhelmed       0.74      0.92      0.82       190
    restless       0.87      0.60      0.71       209

    accuracy                           0.80      1200
   macro avg       0.81      0.80      0.79      1200
weighted avg       0.81      0.80      0.79      1200

Regression
  R2            : -2.1626
  RMSE          : 2.4740
  Exact bin acc : 0.1883
  Adj  bin acc  : 0.3783

[4/4] Inference on test data...

  120 predictions done

  emotional state breakdown:
predic